In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

raw_token = userdata.get("HF_TOKEN")

if not raw_token:
    raise RuntimeError("HF_TOKEN is unavailable.")

token_lines = [
    line.strip()
    for line in raw_token.splitlines()
    if line.strip()
]

if len(token_lines) != 1:
    raise RuntimeError(
        "HF_TOKEN must contain exactly one non-empty line. "
        f"Found {len(token_lines)} lines."
    )

HF_TOKEN = token_lines[0]

if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN does not have the expected format."
    )

if any(character.isspace() for character in HF_TOKEN):
    raise RuntimeError(
        "HF_TOKEN contains whitespace."
    )

login(
    token=HF_TOKEN,
    add_to_git_credential=False,
)

api = HfApi(token=HF_TOKEN)
identity = api.whoami()

print("Token format: valid")
print("Authenticated user:", identity["name"])

In [ ]:
OUTPUT_REPO = "AliothMe/lung-fusion-embeddings"

api.upload_file(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    path_or_fileobj=b"write permission test\n",
    path_in_repo="write_test.txt",
    commit_message="Test Colab write permission",
)

print("Write permission test passed.")

In [ ]:
%pip install -q timm huggingface_hub safetensors pillow numpy

In [ ]:
import huggingface_hub
import timm
import torch

print("torch:", torch.__version__)
print("timm:", timm.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

DATASET_REPO = "AliothMe/lung-fusion-tile-shards"
DATASET_REVISION = (
    "899e7b2c754b8154887f4e818b9482dfbda0c9bc"
)

OUTPUT_REPO = "AliothMe/lung-fusion-embeddings"

UNI2_REPO = "MahmoodLab/UNI2-h"
UNI2_REVISION = (
    "d517a8dd47902dd7c308b3c36f63bce47e7b9a43"
)

WORK_DIR = Path("/content/lung-fusion-agent")
INPUT_DIR = WORK_DIR / "input_shards"
OUTPUT_DIR = WORK_DIR / "uni2_outputs"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input dataset:", DATASET_REPO)
print("Dataset revision:", DATASET_REVISION)
print("Model:", UNI2_REPO)
print("Model revision:", UNI2_REVISION)
print("Output dataset:", OUTPUT_REPO)

In [ ]:
import gc

import timm
import torch
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

device = torch.device("cuda")

torch.set_float32_matmul_precision("high")

globals().pop("model", None)

gc.collect()
torch.cuda.empty_cache()

timm_kwargs = {
    "img_size": 224,
    "patch_size": 14,
    "depth": 24,
    "num_heads": 24,
    "init_values": 1e-5,
    "embed_dim": 1536,
    "mlp_ratio": 2.66667 * 2,
    "num_classes": 0,
    "no_embed_class": True,
    "mlp_layer": timm.layers.SwiGLUPacked,
    "act_layer": torch.nn.SiLU,
    "reg_tokens": 8,
    "dynamic_img_size": True,
}

pinned_model_name = (
    f"hf-hub:{UNI2_REPO}@{UNI2_REVISION}"
)

print("Loading:", pinned_model_name)

model = timm.create_model(
    pinned_model_name,
    pretrained=True,
    **timm_kwargs,
)

transform = create_transform(
    **resolve_data_config(
        model.pretrained_cfg,
        model=model,
    )
)

model = model.eval().to(device)

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Model loaded.")
print("Parameters:", f"{parameter_count:,}")
print("Device:", next(model.parameters()).device)

In [ ]:
import io
import tarfile
import time

import numpy as np
from PIL import Image

BATCH_SIZE = 64
EXPECTED_EMBEDDING_DIM = 1536


def infer_image_batch(
    image_batch: list[torch.Tensor],
) -> np.ndarray:
    batch = torch.stack(image_batch).to(
        device,
        non_blocking=True,
    )

    with torch.inference_mode(), torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):
        batch_embeddings = model(batch)

    batch_embeddings = (
        batch_embeddings
        .to(dtype=torch.float16)
        .cpu()
        .numpy()
    )

    return batch_embeddings


def extract_uni2_shard(
    shard_path: str | Path,
) -> tuple[list[str], np.ndarray, float]:
    tile_ids: list[str] = []
    embedding_batches: list[np.ndarray] = []
    image_batch: list[torch.Tensor] = []

    torch.cuda.reset_peak_memory_stats()
    start_time = time.perf_counter()

    with tarfile.open(shard_path, mode="r") as archive:
        jpg_members = [
            member
            for member in archive.getmembers()
            if member.isfile()
            and member.name.endswith(".jpg")
        ]

        print("JPEG tiles found:", len(jpg_members))

        for tile_number, member in enumerate(
            jpg_members,
            start=1,
        ):
            extracted_file = archive.extractfile(member)

            if extracted_file is None:
                raise RuntimeError(
                    f"Could not extract {member.name}"
                )

            image_bytes = extracted_file.read()

            with Image.open(io.BytesIO(image_bytes)) as image:
                image = image.convert("RGB")
                image_tensor = transform(image)

            tile_ids.append(Path(member.name).stem)
            image_batch.append(image_tensor)

            if len(image_batch) == BATCH_SIZE:
                embedding_batches.append(
                    infer_image_batch(image_batch)
                )
                image_batch.clear()

            if tile_number % 1024 == 0:
                print(
                    f"Processed {tile_number}/"
                    f"{len(jpg_members)} tiles"
                )

        if image_batch:
            embedding_batches.append(
                infer_image_batch(image_batch)
            )
            image_batch.clear()

    torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - start_time

    embeddings = np.concatenate(
        embedding_batches,
        axis=0,
    )

    expected_shape = (
        len(tile_ids),
        EXPECTED_EMBEDDING_DIM,
    )

    if embeddings.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected shape {embeddings.shape}; "
            f"expected {expected_shape}"
        )

    if not np.isfinite(embeddings).all():
        raise RuntimeError(
            "Embeddings contain NaN or infinity."
        )

    if len(set(tile_ids)) != len(tile_ids):
        raise RuntimeError(
            "Duplicate tile IDs found."
        )

    return tile_ids, embeddings, elapsed_seconds

In [ ]:
import hashlib
import json

from huggingface_hub import hf_hub_download


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


def process_and_upload_shard(
    shard_index: int,
) -> dict[str, object]:
    input_name = f"tiles-{shard_index:05d}.tar"
    output_name = f"uni2-{shard_index:05d}.npz"
    metadata_name = f"uni2-{shard_index:05d}.json"

    remote_output_path = f"uni2/{output_name}"
    remote_metadata_path = f"uni2/{metadata_name}"

    print("=" * 60)
    print("Input:", input_name)
    print("Output:", remote_output_path)

    input_path = Path(
        hf_hub_download(
            repo_id=DATASET_REPO,
            filename=input_name,
            repo_type="dataset",
            revision=DATASET_REVISION,
            token=HF_TOKEN,
            local_dir=INPUT_DIR,
        )
    )

    print("Downloaded:", input_path)
    print(
        "Input size:",
        round(input_path.stat().st_size / 1024**2, 2),
        "MiB",
    )

    tile_ids, embeddings, elapsed_seconds = (
        extract_uni2_shard(input_path)
    )

    output_path = OUTPUT_DIR / output_name
    metadata_path = OUTPUT_DIR / metadata_name

    np.savez_compressed(
        output_path,
        tile_ids=np.asarray(tile_ids),
        embeddings=embeddings,
    )

    output_checksum = sha256_file(output_path)

    peak_memory_gib = (
        torch.cuda.max_memory_allocated() / 1024**3
    )

    metadata = {
        "model_name": "UNI2-h",
        "model_repo": UNI2_REPO,
        "model_revision": UNI2_REVISION,
        "dataset_repo": DATASET_REPO,
        "dataset_revision": DATASET_REVISION,
        "input_shard": input_name,
        "output_file": output_name,
        "tile_count": len(tile_ids),
        "embedding_dimension": int(
            embeddings.shape[1]
        ),
        "embedding_dtype": str(embeddings.dtype),
        "batch_size": BATCH_SIZE,
        "elapsed_seconds": elapsed_seconds,
        "tiles_per_second": (
            len(tile_ids) / elapsed_seconds
        ),
        "peak_gpu_memory_gib": peak_memory_gib,
        "output_size_bytes": output_path.stat().st_size,
        "output_sha256": output_checksum,
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": torch.__version__,
        "timm_version": timm.__version__,
    }

    metadata_path.write_text(
        json.dumps(
            metadata,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    print("Local embedding validation:")
    print("  Shape:", embeddings.shape)
    print("  Dtype:", embeddings.dtype)
    print(
        "  Finite:",
        bool(np.isfinite(embeddings).all()),
    )
    print(
        "  Output size:",
        round(output_path.stat().st_size / 1024**2, 2),
        "MiB",
    )

    print("Uploading embedding...")

    api.upload_file(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        path_or_fileobj=str(output_path),
        path_in_repo=remote_output_path,
        commit_message=(
            f"Add UNI2-h shard {shard_index:05d}"
        ),
    )

    print("Uploading metadata...")

    api.upload_file(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        path_or_fileobj=str(metadata_path),
        path_in_repo=remote_metadata_path,
        commit_message=(
            f"Add UNI2-h metadata {shard_index:05d}"
        ),
    )

    print("Upload complete.")
    print(
        f"Speed: "
        f"{len(tile_ids) / elapsed_seconds:.2f} tiles/s"
    )

    return metadata

In [ ]:
required_names = [
    "HF_TOKEN",
    "api",
    "DATASET_REPO",
    "DATASET_REVISION",
    "OUTPUT_REPO",
    "UNI2_REPO",
    "UNI2_REVISION",
    "model",
    "transform",
    "extract_uni2_shard",
    "process_and_upload_shard",
]

for name in required_names:
    status = (
        "ready"
        if name in globals()
        else "MISSING"
    )
    print(f"{name}: {status}")

In [ ]:
pilot_metadata = process_and_upload_shard(0)

print("\n=== Pilot result ===")
print(
    json.dumps(
        pilot_metadata,
        indent=2,
        sort_keys=True,
    )
)

In [ ]:
remote_embedding_path = hf_hub_download(
    repo_id=OUTPUT_REPO,
    filename="uni2/uni2-00000.npz",
    repo_type="dataset",
    token=HF_TOKEN,
    force_download=True,
)

remote_cache = np.load(
    remote_embedding_path,
    allow_pickle=False,
)

remote_tile_ids = remote_cache["tile_ids"]
remote_embeddings = remote_cache["embeddings"]

print("=== Remote validation ===")
print("Tile IDs:", remote_tile_ids.shape)
print("Embeddings:", remote_embeddings.shape)
print("Dtype:", remote_embeddings.dtype)
print(
    "All finite:",
    bool(np.isfinite(remote_embeddings).all()),
)
print(
    "Unique tile IDs:",
    len(set(remote_tile_ids.tolist())),
)
print(
    "Mean L2 norm:",
    round(
        float(
            np.linalg.norm(
                remote_embeddings.astype(np.float32),
                axis=1,
            ).mean()
        ),
        4,
    ),
)

In [ ]:
remote_files = api.list_repo_files(
    OUTPUT_REPO,
    repo_type="dataset",
)

print(
    "Embedding exists:",
    "uni2/uni2-00000.npz" in remote_files,
)
print(
    "Metadata exists:",
    "uni2/uni2-00000.json" in remote_files,
)

In [ ]:

TOTAL_SHARDS = 49
RUN_START_SHARD = 0
RUN_END_SHARD = 49

remote_files = set(
    api.list_repo_files(
        OUTPUT_REPO,
        repo_type="dataset",
    )
)

completed_this_run = []
skipped_this_run = []

full_run_start = time.perf_counter()

for shard_index in range(
    RUN_START_SHARD,
    RUN_END_SHARD,
):
    embedding_path = (
        f"uni2/uni2-{shard_index:05d}.npz"
    )
    metadata_path = (
        f"uni2/uni2-{shard_index:05d}.json"
    )

    embedding_exists = embedding_path in remote_files
    metadata_exists = metadata_path in remote_files

    print("\n" + "=" * 70)
    print(
        f"Shard {shard_index + 1}/{TOTAL_SHARDS}: "
        f"{shard_index:05d}"
    )

    if embedding_exists and metadata_exists:
        print("Remote outputs already exist; skipping.")
        skipped_this_run.append(shard_index)
        continue

    if embedding_exists != metadata_exists:
        print(
            "Partial remote output detected. "
            "The shard will be regenerated."
        )

    shard_metadata = process_and_upload_shard(
        shard_index
    )

    remote_files.add(embedding_path)
    remote_files.add(metadata_path)
    completed_this_run.append(shard_metadata)

    elapsed_total = (
        time.perf_counter() - full_run_start
    )

    finished_count = (
        len(completed_this_run)
        + len(skipped_this_run)
    )

    average_seconds = elapsed_total / max(
        len(completed_this_run),
        1,
    )

    remaining_count = (
        RUN_END_SHARD
        - shard_index
        - 1
    )

    estimated_remaining_minutes = (
        average_seconds
        * remaining_count
        / 60
    )

    print(
        f"Progress: {finished_count}/"
        f"{RUN_END_SHARD - RUN_START_SHARD}"
    )
    print(
        "Estimated remaining time:",
        round(estimated_remaining_minutes, 1),
        "minutes",
    )

full_run_elapsed = (
    time.perf_counter() - full_run_start
)

print("\n=== UNI2-h full run finished ===")
print(
    "Newly completed shards:",
    len(completed_this_run),
)
print(
    "Skipped existing shards:",
    len(skipped_this_run),
)
print(
    "Total elapsed minutes:",
    round(full_run_elapsed / 60, 2),
)

In [ ]:
failed_shard_index = 30

failed_output_path = (
    OUTPUT_DIR / "uni2-00030.npz"
)
failed_metadata_path = (
    OUTPUT_DIR / "uni2-00030.json"
)

print(
    "Embedding exists locally:",
    failed_output_path.exists(),
)
print(
    "Metadata exists locally:",
    failed_metadata_path.exists(),
)

if failed_output_path.exists():
    print(
        "Embedding size:",
        round(
            failed_output_path.stat().st_size
            / 1024**2,
            2,
        ),
        "MiB",
    )

In [ ]:
remote_files = set(
    api.list_repo_files(
        OUTPUT_REPO,
        repo_type="dataset",
    )
)

print(
    "Remote embedding exists:",
    "uni2/uni2-00030.npz" in remote_files,
)
print(
    "Remote metadata exists:",
    "uni2/uni2-00030.json" in remote_files,
)

In [ ]:
import time

from huggingface_hub import HfApi
from huggingface_hub.errors import HfHubHTTPError


def robust_upload_file(*args, **kwargs):
    retry_delays = [0, 10, 20, 40]
    last_error = None

    for attempt_number, delay_seconds in enumerate(
        retry_delays,
        start=1,
    ):
        if delay_seconds:
            print(
                f"Waiting {delay_seconds} seconds "
                "before retry..."
            )
            time.sleep(delay_seconds)

        try:
            result = HfApi.upload_file(
                api,
                *args,
                **kwargs,
            )

            print(
                f"Upload succeeded on attempt "
                f"{attempt_number}."
            )
            return result

        except HfHubHTTPError as error:
            last_error = error

            print(
                f"Upload attempt {attempt_number} "
                f"failed: {type(error).__name__}"
            )

            if attempt_number == len(retry_delays):
                raise

    if last_error is not None:
        raise last_error

    raise RuntimeError("Upload failed without an error.")


api.upload_file = robust_upload_file

print("Upload retry wrapper installed.")

In [ ]:
api.upload_file(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    path_or_fileobj=str(failed_output_path),
    path_in_repo="uni2/uni2-00030.npz",
    commit_message="Retry UNI2-h shard 00030",
)

In [ ]:
api.upload_file(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    path_or_fileobj=str(failed_metadata_path),
    path_in_repo="uni2/uni2-00030.json",
    commit_message="Add UNI2-h metadata 00030",
)

In [ ]:
remote_files = set(
    api.list_repo_files(
        OUTPUT_REPO,
        repo_type="dataset",
    )
)

print(
    "Remote embedding exists:",
    "uni2/uni2-00030.npz" in remote_files,
)
print(
    "Remote metadata exists:",
    "uni2/uni2-00030.json" in remote_files,
)

In [ ]:
remote_30_path = hf_hub_download(
    repo_id=OUTPUT_REPO,
    filename="uni2/uni2-00030.npz",
    repo_type="dataset",
    token=HF_TOKEN,
    force_download=True,
)

remote_30_cache = np.load(
    remote_30_path,
    allow_pickle=False,
)

print(
    "Shape:",
    remote_30_cache["embeddings"].shape,
)
print(
    "Dtype:",
    remote_30_cache["embeddings"].dtype,
)
print(
    "Finite:",
    bool(
        np.isfinite(
            remote_30_cache["embeddings"]
        ).all()
    ),
)
print(
    "Unique tile IDs:",
    len(
        set(
            remote_30_cache[
                "tile_ids"
            ].tolist()
        )
    ),
)

In [ ]:
remote_files = api.list_repo_files(
    OUTPUT_REPO,
    repo_type="dataset",
)

uni2_npz_files = sorted(
    path
    for path in remote_files
    if path.startswith("uni2/")
    and path.endswith(".npz")
)

uni2_json_files = sorted(
    path
    for path in remote_files
    if path.startswith("uni2/")
    and path.endswith(".json")
)

print("UNI2 NPZ files:", len(uni2_npz_files))
print("UNI2 JSON files:", len(uni2_json_files))

print("First NPZ:", uni2_npz_files[0])
print("Last NPZ:", uni2_npz_files[-1])
print("First JSON:", uni2_json_files[0])
print("Last JSON:", uni2_json_files[-1])

In [ ]:
import json

import pandas as pd
from huggingface_hub import hf_hub_download

metadata_records = []

for shard_index in range(49):
    metadata_filename = (
        f"uni2/uni2-{shard_index:05d}.json"
    )

    local_metadata_path = hf_hub_download(
        repo_id=OUTPUT_REPO,
        filename=metadata_filename,
        repo_type="dataset",
        token=HF_TOKEN,
    )

    with open(
        local_metadata_path,
        encoding="utf-8",
    ) as metadata_file:
        metadata_records.append(
            json.load(metadata_file)
        )

uni2_manifest = pd.DataFrame(metadata_records)
uni2_manifest = uni2_manifest.sort_values(
    "input_shard"
).reset_index(drop=True)

print("=== UNI2 metadata summary ===")
print("Records:", len(uni2_manifest))
print(
    "Total tiles:",
    int(uni2_manifest["tile_count"].sum()),
)
print(
    "Embedding dimensions:",
    uni2_manifest[
        "embedding_dimension"
    ].unique(),
)
print(
    "Embedding dtypes:",
    uni2_manifest[
        "embedding_dtype"
    ].unique(),
)
print(
    "Batch sizes:",
    uni2_manifest["batch_size"].unique(),
)
print(
    "Model revisions:",
    uni2_manifest["model_revision"].unique(),
)
print(
    "Dataset revisions:",
    uni2_manifest["dataset_revision"].unique(),
)
print(
    "Total cached size:",
    round(
        uni2_manifest[
            "output_size_bytes"
        ].sum()
        / 1024**3,
        3,
    ),
    "GiB",
)
print(
    "Mean end-to-end speed:",
    round(
        uni2_manifest[
            "tiles_per_second"
        ].mean(),
        2,
    ),
    "tiles/s",
)
print(
    "Min speed:",
    round(
        uni2_manifest[
            "tiles_per_second"
        ].min(),
        2,
    ),
)
print(
    "Max speed:",
    round(
        uni2_manifest[
            "tiles_per_second"
        ].max(),
        2,
    ),
)

In [ ]:
uni2_manifest_path = (
    OUTPUT_DIR / "uni2_manifest.csv"
)

uni2_manifest.to_csv(
    uni2_manifest_path,
    index=False,
)

api.upload_file(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    path_or_fileobj=str(uni2_manifest_path),
    path_in_repo="uni2/manifest.csv",
    commit_message="Add UNI2-h extraction manifest",
)

print("Uploaded: uni2/manifest.csv")

In [ ]:
embedding_repo_info = api.repo_info(
    OUTPUT_REPO,
    repo_type="dataset",
)

print(
    "Embedding repository revision:",
    embedding_repo_info.sha,
)

In [ ]:
EMBEDDING_REVISION = (
    "fe3851889fe76f55c1910ab79e793ff158123147"
)

print("Validating embedding revision:")
print(EMBEDDING_REVISION)

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

tile_index_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename="manifests/tile_to_shard.csv.gz",
    repo_type="dataset",
    revision=DATASET_REVISION,
    token=HF_TOKEN,
)

expected_tile_index = pd.read_csv(
    tile_index_path
)

print("Expected tiles:", len(expected_tile_index))
print(
    "Unique tile IDs:",
    expected_tile_index["tile_id"].nunique(),
)
print(
    "Unique shards:",
    expected_tile_index["shard_name"].nunique(),
)
print(expected_tile_index.head())

In [ ]:
import json
from pathlib import Path

import numpy as np


def calculate_sha256(path: str | Path) -> str:
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        while chunk := file.read(1024 * 1024):
            digest.update(chunk)

    return digest.hexdigest()


validation_records = []
all_observed_tile_ids = set()

total_embedding_rows = 0
total_norm_sum = 0.0
global_norm_min = float("inf")
global_norm_max = float("-inf")

for shard_index in range(49):
    input_shard_name = (
        f"tiles-{shard_index:05d}.tar"
    )
    embedding_filename = (
        f"uni2/uni2-{shard_index:05d}.npz"
    )
    metadata_filename = (
        f"uni2/uni2-{shard_index:05d}.json"
    )

    expected_frame = expected_tile_index.loc[
        expected_tile_index["shard_name"]
        == input_shard_name
    ]

    expected_ids = (
        expected_frame["tile_id"]
        .astype(str)
        .to_numpy()
    )

    embedding_path = hf_hub_download(
        repo_id=OUTPUT_REPO,
        filename=embedding_filename,
        repo_type="dataset",
        revision=EMBEDDING_REVISION,
        token=HF_TOKEN,
    )

    metadata_path = hf_hub_download(
        repo_id=OUTPUT_REPO,
        filename=metadata_filename,
        repo_type="dataset",
        revision=EMBEDDING_REVISION,
        token=HF_TOKEN,
    )

    with open(
        metadata_path,
        encoding="utf-8",
    ) as metadata_file:
        shard_metadata = json.load(metadata_file)

    with np.load(
        embedding_path,
        allow_pickle=False,
    ) as cache:
        observed_ids = (
            cache["tile_ids"]
            .astype(str)
        )
        observed_embeddings = cache["embeddings"]

        expected_shape = (
            len(expected_ids),
            1536,
        )

        if observed_embeddings.shape != expected_shape:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"shape {observed_embeddings.shape}, "
                f"expected {expected_shape}"
            )

        if observed_embeddings.dtype != np.float16:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"unexpected dtype "
                f"{observed_embeddings.dtype}"
            )

        if not np.isfinite(
            observed_embeddings
        ).all():
            raise RuntimeError(
                f"{embedding_filename}: "
                "contains NaN or infinity"
            )

        if len(np.unique(observed_ids)) != len(
            observed_ids
        ):
            raise RuntimeError(
                f"{embedding_filename}: "
                "duplicate tile IDs within shard"
            )

        if not np.array_equal(
            observed_ids,
            expected_ids,
        ):
            observed_set = set(
                observed_ids.tolist()
            )
            expected_set = set(
                expected_ids.tolist()
            )

            missing = expected_set - observed_set
            extra = observed_set - expected_set

            raise RuntimeError(
                f"{embedding_filename}: tile IDs "
                "do not match input index; "
                f"missing={len(missing)}, "
                f"extra={len(extra)}"
            )

        overlap = (
            all_observed_tile_ids
            & set(observed_ids.tolist())
        )

        if overlap:
            raise RuntimeError(
                f"{embedding_filename}: "
                f"{len(overlap)} tile IDs already "
                "appeared in another shard"
            )

        all_observed_tile_ids.update(
            observed_ids.tolist()
        )

        embedding_float32 = (
            observed_embeddings.astype(
                np.float32
            )
        )
        norms = np.linalg.norm(
            embedding_float32,
            axis=1,
        )

        total_embedding_rows += len(
            observed_embeddings
        )
        total_norm_sum += float(norms.sum())
        global_norm_min = min(
            global_norm_min,
            float(norms.min()),
        )
        global_norm_max = max(
            global_norm_max,
            float(norms.max()),
        )

    observed_checksum = calculate_sha256(
        embedding_path
    )
    expected_checksum = shard_metadata[
        "output_sha256"
    ]

    if observed_checksum != expected_checksum:
        raise RuntimeError(
            f"{embedding_filename}: "
            "SHA-256 mismatch"
        )

    validation_records.append(
        {
            "shard_index": shard_index,
            "input_shard": input_shard_name,
            "embedding_file": embedding_filename,
            "tile_count": len(expected_ids),
            "embedding_dimension": 1536,
            "embedding_dtype": "float16",
            "all_finite": True,
            "tile_order_matches": True,
            "sha256_matches": True,
            "mean_l2_norm": float(norms.mean()),
            "min_l2_norm": float(norms.min()),
            "max_l2_norm": float(norms.max()),
        }
    )

    print(
        f"[{shard_index + 1:02d}/49] "
        f"{embedding_filename}: "
        f"{len(expected_ids)} tiles ✓"
    )

In [ ]:
validation_frame = pd.DataFrame(
    validation_records
)

validation_output_path = (
    OUTPUT_DIR / "uni2_validation.csv"
)

validation_frame.to_csv(
    validation_output_path,
    index=False,
)

api.upload_file(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    path_or_fileobj=str(
        validation_output_path
    ),
    path_in_repo="uni2/validation.csv",
    commit_message=(
        "Add UNI2-h embedding validation report"
    ),
)

final_embedding_info = api.repo_info(
    OUTPUT_REPO,
    repo_type="dataset",
)

print(
    "Validation report uploaded:",
    "uni2/validation.csv",
)
print(
    "Final embedding revision:",
    final_embedding_info.sha,
)